## Rollout history and rollback

Each rollout creates a new ReplicaSet, and the Deployment keeps a numbered history.

```bash
kubectl rollout history deployment/web
# REVISION  CHANGE-CAUSE
# 1         <none>
# 2         <none>
kubectl rollout history deployment/web --revision=2   # inspect one revision's template
```

### Roll back

One command reverts:

```bash
kubectl rollout undo deployment/web                 # to the previous revision
kubectl rollout undo deployment/web --to-revision=1 # to a specific one
```

`undo` is *just another rollout* — it creates a **new** revision whose template equals the old one, then runs the same rolling process. So after undo you see revisions 1, 2, 3 — where 3's template matches 1's.

### Pause / resume — batching risky changes

```bash
kubectl rollout pause deployment/web    # further template edits won't trigger a rollout
# ...batch several changes...
kubectl rollout resume deployment/web   # all ship as one rollout
```

### The change-cause breadcrumb

The `CHANGE-CAUSE` column reads the `kubernetes.io/change-cause` annotation. Set it to leave notes for future-you:

```bash
kubectl annotate deployment/web \
  kubernetes.io/change-cause="bump nginx to 1.28 for CVE-2025-xxxx"
```

Any revision created *after* the annotation inherits that cause. On our map this is the **rollout** verb on the left driving the **Deployment** — the history of ReplicaSets is what makes "go back to last week" a single command.